In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_parquet("../data/all_clean.parquet")

In [4]:
#VelTime => 362
#VelTrace => 362 
vcols = [c for c in df.columns if c.startswith("VelTrace_")]
tcols = [c for c in df.columns if c.startswith("VelTime_")]

first_v_index = df.columns.get_loc(vcols[0])
last_v_index = df.columns.get_loc(vcols[-1])
first_t_index = df.columns.get_loc(tcols[0])
last_t_index = df.columns.get_loc(tcols[-1])


In [40]:
threshold = 1e-3

rt1_list = []
rt2_list = []
rt1_status_list = []
bad_trials = []

for i in range(len(df)):
    
    v = df.iloc[i, first_v_index:last_v_index].to_numpy(dtype=float)
    t = df.iloc[i, first_t_index:last_t_index].to_numpy(dtype=float)

    correct = df.iloc[i]["Correct"]

    # Remove NaN values
    valid = np.isfinite(v) & np.isfinite(t)
    v = v[valid]
    t = t[valid]

    reward_time = df.iloc[i]["ReactionTime"]

    if pd.notna(reward_time):
        mask = t <= reward_time
        t = t[mask]
        v = v[mask]

    # Remove backward time values:
    keep = np.ones(len(t), dtype=bool)
    max_time = t[0]
    for i in range(1, len(t)):
        if t[i] > max_time:
            max_time = t[i]
        else:
            keep[i] = False
    
    t = t[keep]
    v = v[keep]


                                                        # =========================
                                                        # ====== RUNNING RAT ======
                                                        # =========================
    rt1 = np.nan
    rt2 = np.nan
    status = ''
    if correct in [1, 3]:

        # Method 1: last upward threshold crossing
        crossings = np.where((v[:-1] <= threshold) &(v[1:] > threshold))[0] + 1
        if len(crossings) > 0:
            rt1 = t[crossings[-1]]
            status = 'cross_up'
        else:
            rt1 = 0.0
            status = 'already running'

        # Method 2: peak positive acceleration
        if len(v) >= 2:
            
            if len(t) < 2:
                bad_trials.append((i, "too_few_points"))
                continue
            dt = np.diff(t)
            if np.any(dt <= 0):
                bad_trials.append((i, "duplicate_or_decreasing_time"))
                continue
                
            acc = np.gradient(v, t)
            peak_acc_idx = np.nanargmax(acc)
            rt2 = t[peak_acc_idx]
            
                                                        # ========================
                                                        # ====== STILL RAT =======
                                                        # ========================
    elif correct in [0, 2]:

        # Method 1: last downward threshold crossing
        crossings = np.where((v[:-1] >= threshold) &(v[1:] < threshold))[0] + 1

        if len(crossings) > 0:
            rt1 = t[crossings[-1]]
            status = 'cross_down'
        else:
            rt1 = -999
            status = 'never_moved'

        # Method 2: peak negative acceleration
        if len(v) >= 2:
            acc = np.gradient(v, t)
            peak_deacc_idx = np.nanargmin(acc)
            rt2 = t[peak_deacc_idx]

    rt1_list.append(rt1)
    rt2_list.append(rt2)
    rt1_status_list.append(status)






In [41]:
df["rt_thr"] = rt1_list
df["rt_acc"] = rt2_list
df["rt_thr_status"] = rt1_status_list

In [43]:
df.to_parquet('../data/rt1rt2all.parquet')

In [27]:
i = 2
df.loc[1,'RatID']

np.int64(10501)

In [28]:
bad_trials = []

for i in range(len(df)):
    
    v = df.iloc[i, first_v_index:last_v_index].to_numpy(dtype=float)
    t = df.iloc[i, first_t_index:last_t_index].to_numpy(dtype=float)

    # Remove NaN values
    valid = np.isfinite(v) & np.isfinite(t)
    v = v[valid]
    t = t[valid]
            
   
    dt = np.diff(t)
    if np.any(dt <= 0):
        bad = np.where(dt <= 0)[0]
        bad_trials.append({
            "RatID": df.loc[i, "RatID"],
            "Date": df.loc[i, "Date"],
            "Trial": df.loc[i, "Trial"],
            "Row": i,
            "BadIndices": bad.tolist()
        })
        continue

            
                                                    
    



In [33]:
ff = pd.DataFrame(bad_trials)

In [36]:
ff.to_csv('../data/DecreasingTimeTrials.csv')

In [41]:
ff

,RatID,Date,Trial,Row,BadIndices
0,291,010620,63,86034,[103]
1,291,010620,95,86066,[239]
2,291,010620,135,86106,[309]
3,291,010620,247,86218,[279]
4,291,020620,344,86815,[307]
...,...,...,...,...,...
344,8801,170319,441,641427,[63]
345,9152,110219,146,666332,"[24, 27]"
346,9152,110219,216,666402,"[294, 297]"
347,9152,110219,217,666403,"[263, 266]"


In [50]:
ff.groupby(['RatID','Date'])['Trial'].unique()


RatID  Date  
291    010620                          [63, 95, 135, 247]
       020620                                       [344]
       070620                             [124, 416, 480]
       080620    [126, 153, 180, 206, 226, 338, 435, 444]
       090620                                       [175]
                                   ...                   
8451   210918                                        [21]
       220918                                    [8, 477]
       290918                                       [122]
8801   170319                                       [441]
9152   110219                        [146, 216, 217, 297]
Name: Trial, Length: 142, dtype: object

### Strip down the t and v cols so you cut all the NaNs. I ended up having vel and t trace until _345 index.

In [5]:
#df = pd.read_parquet("../data/rt1rt2all.parquet")

# Get the time and velocity column names before modifying anything
t_cols = df.columns[first_t_index:last_t_index]
v_cols = df.columns[first_v_index:last_v_index]

# Number of valid paired samples in each trial
valid_lengths = (df[t_cols].notna().to_numpy()& df[v_cols].notna().to_numpy()).sum(axis=1)

# Shortest trial length
min_length = valid_lengths.min()

print("Minimum valid length:", min_length)

# Columns to remove
t_cols_to_drop = t_cols[min_length:]
v_cols_to_drop = v_cols[min_length:]

# Cut the full dataset
df = df.drop(columns=list(t_cols_to_drop) + list(v_cols_to_drop))


# Save the shortened full dataset
df.to_parquet("../data/rt1rt2all_cut.parquet",index=False)

print("Full data shape:", df.shape)

Minimum valid length: 345
Full data shape: (709987, 706)
